<table dir="ltr" width="100%"><thead><tr><th width="50%" dir="ltr" lang="en" align="left">English</th><th width="50%" dir="rtl" lang="ar" align="right">العربية</th></tr></thead><tbody><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h1>Lab 02 preparation · Cost arithmetic</h1><p>Compare two operation policies using hypothetical teaching units (TU). This is not a quotation, currency forecast or Spark runtime benchmark.</p></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h1>تحضير اللاب 02 · حساب التكلفة</h1><p>قارن سياستي تشغيل بوحدات تعليم افتراضية TU. هذا ليس عرض سعر أو توقع عملة أو قياسًا لأداء Spark.</p></td></tr></tbody></table>

<table dir="ltr" width="100%"><thead><tr><th width="50%" dir="ltr" lang="en" align="left">English</th><th width="50%" dir="rtl" lang="ar" align="right">العربية</th></tr></thead><tbody><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h2>Goal and setup</h2><p>Show how work hours, startup and overhead affect the result; do not assume scheduled compute is always cheaper. Shared code is in <code>src/masar/cost.py</code>.</p></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h2>الهدف والإعداد</h2><p>بيّن أثر ساعات العمل والبدء والتكاليف الإضافية؛ ولا تفترض أن الحوسبة المجدولة أرخص دائمًا. الكود المشترك في <code dir="ltr">src/masar/cost.py</code>.</p></td></tr></tbody></table>

In [ ]:
from pathlib import Path
import sys, json, tempfile
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "course.json").is_file() and (p / "src" / "masar").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook from within the complete course repository.")
sys.path.insert(0, str(ROOT / "src"))
(ROOT / "outputs").mkdir(exist_ok=True)
RUN = Path(tempfile.mkdtemp(prefix="day01_", dir=ROOT / "outputs"))
print("Scope: preparatory Python helper only; Spark/Delta are not executed.")

<table dir="ltr" width="100%"><thead><tr><th width="50%" dir="ltr" lang="en" align="left">English</th><th width="50%" dir="rtl" lang="ar" align="right">العربية</th></tr></thead><tbody><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h2>1. Inspect assumptions</h2><p>The base horizon is one 30-day teaching month. The storage charge is a monthly assumption; keep the horizon fixed when interpreting this exercise.</p></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h2>1. افحص الافتراضات</h2><p>الفترة الأساسية شهر تعليمي من 30 يومًا. تكلفة التخزين افتراض شهري؛ حافظ على هذه الفترة عند تفسير التمرين.</p></td></tr></tbody></table>

In [ ]:
from decimal import Decimal
from masar.cost import DEFAULTS, evaluate_cost, serializable, cost_report
print(json.dumps(DEFAULTS, indent=2))
result = cost_report()
print(json.dumps(result["base"], indent=2))

<table dir="ltr" width="100%"><thead><tr><th width="50%" dir="ltr" lang="en" align="left">English</th><th width="50%" dir="rtl" lang="ar" align="right">العربية</th></tr></thead><tbody><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h2>2. Test workload sensitivity</h2><p>Only work hours change below. The remaining assumptions stay fixed so the comparison is interpretable.</p></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h2>2. اختبر حساسية حمل العمل</h2><p>تتغير ساعات العمل فقط أدناه؛ وتبقى بقية الافتراضات ثابتة لتكون المقارنة قابلة للتفسير.</p></td></tr></tbody></table>

In [ ]:
print("Work h/day | Always-on TU | Scheduled TU | Difference TU")
for row in result["sensitivity"]:
    print(f"{row['work_hours_per_day']:>10} | {row['always_on_total']:>12} | {row['scheduled_total']:>12} | {row['difference']:>13}")

<table dir="ltr" width="100%"><thead><tr><th width="50%" dir="ltr" lang="en" align="left">English</th><th width="50%" dir="rtl" lang="ar" align="right">العربية</th></tr></thead><tbody><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h2>3. Check boundaries and invalid inputs</h2><p>Test equality, a counterexample, zero rates and a rejected invalid input. Passing arithmetic does not establish cloud economics or real performance.</p></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h2>3. افحص الحدود والمدخلات غير الصالحة</h2><p>اختبر التعادل والمثال المضاد والمعدل الصفري ورفض مدخل غير صالح. نجاح الحساب لا يثبت اقتصاديات سحابية أو أداءً فعليًا.</p></td></tr></tbody></table>

In [ ]:
base = evaluate_cost(DEFAULTS)
checks = {
    "base_totals": (base["always_on_total"], base["scheduled_total"]) == (Decimal("1460"), Decimal("185")),
    "break_even": evaluate_cost({**DEFAULTS, "work_hours_per_day": "23.25"})["difference"] == 0,
    "counterexample": evaluate_cost({**DEFAULTS, "work_hours_per_day": "23.75"})["difference"] < 0,
    "zero_rate": evaluate_cost({**DEFAULTS, "price_per_core_hour": "0"})["break_even_work_hours_per_day"] is None,
}
try:
    evaluate_cost({**DEFAULTS, "cores": "-1"})
except ValueError:
    checks["negative_input_rejected"] = True
else:
    checks["negative_input_rejected"] = False
if not all(checks.values()):
    raise AssertionError(checks)
result["checks"] = checks
print(json.dumps(checks, indent=2))

<table dir="ltr" width="100%"><thead><tr><th width="50%" dir="ltr" lang="en" align="left">English</th><th width="50%" dir="rtl" lang="ar" align="right">العربية</th></tr></thead><tbody><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h2>4. Save evidence</h2><p>The artifact explicitly records exclusions. Keep real Spark measurements separate when the full lab is available.</p></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h2>4. احفظ الأدلة</h2><p>يسجل الملف ما يستبعده الحساب صراحة. افصل قياسات Spark الفعلية عندما يتاح اللاب الكامل.</p></td></tr></tbody></table>

In [ ]:
output = RUN / "cost_model_result.json"
output.write_text(json.dumps(result, sort_keys=True, indent=2) + "\n", encoding="utf-8")
print("PASS: hypothetical cost arithmetic only")
print("Saved:", output.name)
print("Spark benchmark: NOT EXECUTED")

<table dir="ltr" width="100%"><thead><tr><th width="50%" dir="ltr" lang="en" align="left">English</th><th width="50%" dir="rtl" lang="ar" align="right">العربية</th></tr></thead><tbody><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h2>Interpretation and next step</h2><p>Explain the break-even workload and one excluded cost. Do not label TU as SAR or claim a measured speed-up. Continue with the Lab 02 contract when the actual benchmark is verified.</p></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h2>التفسير والخطوة التالية</h2><p>اشرح حمل التعادل وتكلفة مستبعدة واحدة. لا تسم وحدات TU ريالات ولا تدع تسارعًا مقاسًا. أكمل مواصفات اللاب 02 عند التحقق من القياس الفعلي.</p></td></tr></tbody></table>